# Module 3 — Robust Evaluation

**By the end of this notebook, you will be able to:**
- Explain why a single manual train/test pair gives a fragile, direction-dependent performance estimate
- Use `GroupKFold` to evaluate a model against every group in turn, without ever mixing a group across train and validation
- Read a mean **and** a std across folds, not just a single number

**Context:** In Session 2's reflection, you compared training on Kampala and testing on Nairobi, then reversed it — and got two different, both disappointing results. That comparison only used two of the four cities, and only one direction each. This module systematizes it: evaluate against every city, one at a time, on the full 4-city dataset you cleaned in Module 2.

## Rebuild the cleaned dataset

**Exercise:** Reuse the functions you completed in Module 2 to rebuild `enriched`: scope `train_df` to all four cities and `DEFAULT_COLUMNS`, drop columns above a 0.7 missing-value threshold, fill what remains per city, then add temporal features — the exact same sequence as `workflows.run_advanced`, minus the final split.

In [ ]:
# TODO: rebuild enriched, reusing the Module 2 functions you already completed


## One pair only tells you about that pair

Recall Session 2's numbers:

| Direction | RMSE | R² |
|---|---|---|
| Train Kampala → test Nairobi | 24.6 | -0.02 |
| Train Nairobi → test Kampala | 15.2 | -0.16 |

Neither number tells you anything about Lagos or Bujumbura — they never appeared in either run. Would the model do better or worse there? You cannot know from a single manual pair, and picking a *different* pair would have given you yet another, equally partial answer.

[`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) automates the *idea* behind the manual split — never mix a city across train and validation — but applies it systematically: with one fold per city, each city takes a turn being the one the model has never seen, while training on the other three. This is sometimes called leave-one-group-out.

## Implement it

Open `src/air_quality/evaluation.py` and complete `evaluate_group_cv`, in the "Session 3 — optional modules" section at the bottom of the file. Its docstring specifies the exact contract, and `tests/test_evaluation_advanced.py` specifies the exact expected behavior — read it the same way you read `tests/test_evaluation.py` in Session 2. One detail the docstring calls out: fit a fresh [`clone()`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html) of the model for each fold, not the same instance refit repeatedly — a fold must never carry over anything learned by a previous fold. Run `uv run pytest tests/test_evaluation_advanced.py -v` until it passes, then come back here.

## Apply it, on all four cities at once

**Exercise:** Call `evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")`, then look at the per-fold breakdown as a table.

In [ ]:
# Given: once evaluate_group_cv is implemented, this just calls it
result = evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")

pd.DataFrame(result["folds"]).set_index("test_group")

In [ ]:
# Given: the aggregated view
{k: v for k, v in result.items() if k != "folds"}

**Look at the per-city RMSE.** One city is far harder to predict for than the other three — the kind of gap a single Kampala/Nairobi comparison could never have revealed, in either direction.

**Question:** Which city has the highest RMSE when held out? Does the mean RMSE across all four folds match either of Session 2's two numbers (24.6 or 15.2)? What does a large std across folds tell you about how much you should trust a performance number computed from a single train/test pair, the way Session 2 did?

## From notebook to pipeline

This replaces part of what `run_advanced` (in `src/air_quality/workflows.py`) already does after Module 2: keep the same cleaning (scope, drop high-missing columns, fill, add temporal features), but replace the manual-split evaluation at the end with `evaluate_group_cv` across every city. `config.train_city`/`config.test_city` are no longer needed at this point — feel free to remove them from `AdvancedPipelineConfig` too. As with Module 2, there is no test for this: verify it by calling it below and comparing to what you got above.

In [ ]:
# Once you have updated run_advanced in workflows.py, this should print
# the same folds and aggregates you got above
from air_quality.workflows import run_advanced

run_advanced()

## Wrap-up

Write down: which city was hardest to generalize to, and in one sentence, why a mean *and* a std across folds tells you more than the single number Session 2's manual split gave you.

_Your observations here._